In [1]:
import pandas as pd
from google.colab import files
uploaded = files.upload()

Saving realty_data[1].csv to realty_data[1].csv


In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib

# Загрузка данных
data = pd.read_csv("realty_data[1].csv")

In [3]:
data.head()

,product_name,period,price,postcode,address_name,lat,lon,object_type,total_square,rooms,floor,city,settlement,district,area,description,source
0,"3-комнатная, 137 м²",NaN,63000000,127473.0,"2-й Щемиловский переулок, 5а",55.778894,37.608844,Квартира,137.0,3.0,6.0,Москва,NaN,Тверской район,NaN,Просторная квартира свободной планировки с пан...,ЦИАН
1,"Студия, 16,7 м²",NaN,3250000,108815.0,"Харлампиева, 46",55.551025,37.313054,Квартира,16.7,NaN,1.0,Москва,NaN,Филимонковское поселение,NaN,ВНИМАНИЕ! ОЧЕНЬ ПРИВЛЕКАТЕЛЬНОЕ ПРЕ...,Домклик
2,"3-комнатная, 76 м²",NaN,16004680,NaN,"ЖК Прокшино, 8 к4",55.594802,37.431264,Квартира,76.0,3.0,6.0,Москва,NaN,Сосенское поселение,NaN,"Apт.1684018. 0,01% - гибкая ипотека! Воспользу...",Яндекс.Недвижимость
3,"1-комнатная, 24 м²",NaN,7841776,NaN,"ЖК Прокшино, 6 к2",55.594332,37.428099,Квартира,24.0,1.0,10.0,Москва,NaN,Сосенское поселение,NaN,Продается однокомнатная квартира № 381 в новос...,Новострой-М
4,"3-комнатная, 126 м²",NaN,120000000,121352.0,"Давыдковская, 18",55.721097,37.464342,Квартира,126.0,3.0,16.0,Москва,NaN,Фили-Давыдково район,NaN,Шикарное предложение!\nПродаётся трёхкомнатная...,Домклик


In [4]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from sklearn.linear_model import LinearRegression
import numpy as np

In [10]:


data = pd.read_csv("realty_data[1].csv")

target_col = "price"

numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
if target_col in numeric_cols:
    numeric_cols.remove(target_col)

X = data[numeric_cols]
y = data[target_col]

# Обработка пропусков
X_filled = X.fillna(X.median())
cols_to_drop = X_filled.columns[X_filled.isna().any()].tolist()
if cols_to_drop:
    print("Удаляем колонки с пропусками:", cols_to_drop)
    X_filled = X_filled.drop(columns=cols_to_drop)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_filled, y, test_size=0.2, random_state=42)

# Масштабирование
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Обучение
model = LinearRegression()
model.fit(X_train_scaled, y_train)

score = model.score(X_test_scaled, y_test)
print(f"R^2 на тестовой выборке: {score:.3f}")

# Сохранение
joblib.dump(model, "model.pkl")
joblib.dump(scaler, "scaler.pkl")

feature_info = []
for col in X_filled.columns:
    feature_info.append({
        "name": col,
        "min": float(X_filled[col].min()),
        "max": float(X_filled[col].max()),
        "default": float(X_filled[col].median())
    })
joblib.dump(feature_info, "feature_info.pkl")
print("Файлы модели сохранены: model.pkl, scaler.pkl, feature_info.pkl")


Удаляем колонки с пропусками: ['period']
R^2 на тестовой выборке: 0.750
Файлы модели сохранены: model.pkl, scaler.pkl, feature_info.pkl


In [15]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
import numpy as np
import joblib

# --- Загрузка артефактов ---
try:
    model = joblib.load("model.pkl")
    scaler = joblib.load("scaler.pkl")
    feature_info = joblib.load("feature_info.pkl")
except FileNotFoundError as e:
    raise RuntimeError(
        "Не найдены файлы модели. Сначала запусти скрипт обучения, чтобы создать model.pkl, scaler.pkl, feature_info.pkl."
    ) from e

app = FastAPI(title="Real Estate Price Prediction API")

feature_names = [f["name"] for f in feature_info]

# --- Динамическая модель для POST-запроса ---
class DynamicPredictionInput(BaseModel):
    pass

for info in feature_info:
    name = info["name"]
    min_val = info["min"]
    max_val = info["max"]

    field_kwargs = {}
    if min_val is not None:
        field_kwargs["ge"] = min_val
    if max_val is not None:
        field_kwargs["le"] = max_val

    # Обязательное поле: только ... без default
    setattr(DynamicPredictionInput, name, Field(..., **field_kwargs))


class PredictionResponse(BaseModel):
    predicted_price: float
    features_used: list[str]


class HealthResponse(BaseModel):
    status: str
    service: str


# --- Liveness-проба (health-check) ---
@app.get("/health", response_model=HealthResponse)
def health():
    return {"status": "healthy", "service": "price-prediction-api"}


# --- GET /predict_get ---
@app.get("/predict_get", response_model=PredictionResponse)
def predict_get(**kwargs):
    input_values = []
    for info in feature_info:
        name = info["name"]
        if name not in kwargs:
            raise HTTPException(status_code=400, detail=f"Отсутствует обязательный признак: {name}")
        val = kwargs[name]
        try:
            val = float(val)
        except (TypeError, ValueError):
            raise HTTPException(status_code=400, detail=f"Значение '{name}' должно быть числом")
        if val < info["min"] or val > info["max"]:
            raise HTTPException(
                status_code=400,
                detail=f"Значение '{name}'={val} выходит за диапазон [{info['min']}, {info['max']}]"
            )
        input_values.append(val)

    X_input = np.array([input_values])
    X_scaled = scaler.transform(X_input)
    pred = model.predict(X_scaled)[0]

    return {
        "predicted_price": float(pred),
        "features_used": feature_names
    }


# --- POST /predict_post ---
@app.post("/predict_post", response_model=PredictionResponse)
def predict_post(payload: DynamicPredictionInput):
    input_values = [getattr(payload, name) for name in feature_names]
    X_input = np.array([input_values])
    X_scaled = scaler.transform(X_input)
    pred = model.predict(X_scaled)[0]

    return {
        "predicted_price": float(pred),
        "features_used": feature_names
    }
